<a href="https://colab.research.google.com/github/BensonAdomako/Thesis/blob/main/Yield_Predictions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Extract + Save Results

In [ ]:
import pandas as pd
import joblib
import numpy as np
import os
import glob
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define the directory where models and data were saved
model_dir = "/content/drive/MyDrive/Crop Yield Prediction/models_gpu_spatialcv"

# 1. Load the data
panel_path = os.path.join(model_dir, "EE_harvest_ml_full_panel.csv")
if os.path.exists(panel_path):
    df_panel = pd.read_csv(panel_path)
    print(f"✅ Loaded data from {panel_path}")
    print(f"Shape: {df_predictions.shape}")
else:
    raise FileNotFoundError(f"File not found: {panel_path}. Please check if the training cell ran successfully.")

# 2. Load ALL models (Global + Country-specific)
model_files = glob.glob(os.path.join(model_dir, "model_*.joblib"))
models = {}

print("\n🔹 Loading models...")
for mp in model_files:
    # Extract name: model_GLOBAL.joblib -> GLOBAL
    fname = os.path.basename(mp)
    model_name = fname.replace("model_", "").replace(".joblib", "")

    try:
        loaded_model = joblib.load(mp)
        models[model_name] = loaded_model
        print(f"  - Loaded {model_name}")
    except Exception as e:
        print(f"  ⚠️ Failed to load {model_name}: {e}")

# 3. Define feature variables
# Exclude metadata, targets, and existing prediction columns
merge_vars = ['country','year','lat_modified','lon_modified', 'harvest_end_month', 'harvest_end_month_dt']
exclude_cols = merge_vars
x_vars = [c for c in df_panel.columns
          if c not in exclude_cols
          and not c.startswith('pred_')
          and not c.startswith('Unnamed')]

X_loaded = df_panel[x_vars]
print(f"\nFeatures selected: {len(x_vars)}")

# Convert object -> numeric safely (important!)
for c in X_loaded.columns:
    if X_loaded[c].dtype == "object":
        X_loaded[c] = pd.to_numeric(X_loaded[c], errors="coerce")

# Median impute
X_loaded = X_loaded.fillna(X_loaded.median(numeric_only=True))

# Fill any still-all-NA columns with 0
still_na_cols = X_loaded.columns[X_loaded.isna().any()].tolist()
if still_na_cols:
    print("⚠️ Still-NA columns after median fill (likely all missing). Filling with 0.")
    X_loaded[still_na_cols] = X_loaded[still_na_cols].fillna(0)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Loaded data from /content/drive/MyDrive/Crop Yield Prediction/models_gpu_spatialcv/EE_harvest_ml_full_panel.csv
Shape: (98280, 90)

🔹 Loading models...
  - Loaded Uganda
  - Loaded GLOBAL
  - Loaded Ethiopia
  - Loaded Malawi
  - Loaded Mali
  - Loaded Niger
  - Loaded Nigeria
  - Loaded Tanzania

Features selected: 84


In [ ]:
# ----------------------------------------------------------
# 4) Predict: use ONLY country-specific models (NO fallback)
# ----------------------------------------------------------
print("\n🔹 Predicting with country-specific models ONLY (no GLOBAL fallback)...")

pred_log_final = np.full(len(df_panel), np.nan, dtype=float)

for ctry in sorted(df_panel['country'].astype(str).unique()):
    mask = (df_panel['country'].astype(str) == ctry)
    n = int(mask.sum())

    if ctry in models:
        pred_log_final[mask] = models[ctry].predict(X_loaded.loc[mask])
        print(f"  ✅ {ctry}: predicted {n} rows with country model")
    else:
        print(f"  ❌ {ctry}: no country model → predictions left as NA ({n} rows)")

df_panel["pred_log_yield"] = pred_log_final
df_panel["pred_yield"] = np.expm1(df_panel["pred_log_yield"])

# ----------------------------------------------------------
# 5) Keep only necessary columns
# ----------------------------------------------------------
cols_to_keep = [
    "country",
    "year",
    "lat_modified",
    "lon_modified",
    "harvest_end_month",
    "pred_log_yield",
    "pred_yield"
]

missing = [c for c in cols_to_keep if c not in df_panel.columns]
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

df_final = df_panel[cols_to_keep].copy()

print("Final output shape:", df_final.shape)
print(df_final.head())

output_path = "/content/drive/MyDrive/Crop Yield Prediction/predicted_yields.csv"

df_final.to_csv(output_path, index=False)

print(f"✅ Saved final predicted yield panel to:\n{output_path}")




🔹 Predicting with country-specific models ONLY (no GLOBAL fallback)...
  ✅ Ethiopia: predicted 11685 rows with country model
  ✅ Malawi: predicted 24975 rows with country model
  ✅ Mali: predicted 12345 rows with country model
  ✅ Niger: predicted 2400 rows with country model
  ✅ Nigeria: predicted 12825 rows with country model
  ✅ Tanzania: predicted 26310 rows with country model
  ✅ Uganda: predicted 7740 rows with country model
Final output shape: (98280, 7)
    country  year  lat_modified  lon_modified harvest_end_month  \
0  Ethiopia  2010      3.455701     39.515994        2010-01-01   
1  Ethiopia  2010      3.549937     39.184234        2010-01-01   
2  Ethiopia  2010      3.982931     38.491368        2010-12-01   
3  Ethiopia  2010      4.281844     41.875076        2010-01-01   
4  Ethiopia  2010      4.744136     36.045393        2010-01-01   

   pred_log_yield   pred_yield  
0        7.030554  1129.657154  
1        7.015233  1112.465567  
2        6.824152   918.795641 

In [ ]:
df_final.groupby("country").agg(
    n_obs=("pred_yield", "size"),
    min_year=("year", "min"),
    max_year=("year", "max")
)


,n_obs,min_year,max_year
country,,,
Ethiopia,11685,2010,2024
Malawi,24975,2010,2024
Mali,12345,2010,2024
Niger,2400,2010,2024
Nigeria,12825,2010,2024
Tanzania,26310,2010,2024
Uganda,7740,2010,2024
